<a href="https://colab.research.google.com/github/malkunn/Hourly-River-Streamflow-Forecasting-Hackathon./blob/main/StreamFlowForecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

# 1. Load Files
df_train = pd.concat([pd.read_csv('trainTrackA.csv'), pd.read_csv('trainTrackB.csv')]).reset_index(drop=True)
df_test = pd.concat([pd.read_csv('testTrackA (2).csv'), pd.read_csv('testTrackB (2).csv')]).reset_index(drop=True)
example_sub = pd.read_csv('ExampleSubmission(1).csv')

# 2. Convert Dates
df_train['date'] = pd.to_datetime(df_train['date'])
df_test['date'] = pd.to_datetime(df_test['date'])

# Identify features and target
target_col = 'qobs_mm_per_hour'
features = [col for col in df_train.columns if col not in ['date', 'gauge_id', 'Unique_ID', target_col]]

print(f"Test data starts on: {df_test['date'].min()}") # Should be Dec 28, 2017

Test data starts on: 2017-01-01 00:00:00


In [33]:
# 1. Clean Data (ffill/bfill prevents NaNs)
df_train = df_train.ffill().bfill().fillna(0)
df_test = df_test.ffill().bfill().fillna(0)

# 2. Scaling
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

train_x_scaled = scaler_x.fit_transform(df_train[features])
train_y_scaled = scaler_y.fit_transform(df_train[[target_col]])

# 3. Create training sequences
def create_sequences(data_x, data_y, window_size=24, forecast_size=4):
    X, y = [], []
    for i in range(len(data_x) - window_size - forecast_size + 1):
        X.append(data_x[i : i + window_size])
        y.append(data_y[i + window_size : i + window_size + forecast_size].flatten())
    return np.array(X), np.array(y)

WINDOW_SIZE = 24
FORECAST_SIZE = 4

X_train, y_train = create_sequences(train_x_scaled, train_y_scaled, WINDOW_SIZE, FORECAST_SIZE)

ValueError: could not convert string to float: 'Carbonate sedimentary rocks'

In [34]:
# 1. Fill missing values (NaNs)
# We use 'ffill' (forward fill) to take the last known value, then 'bfill' for anything left
df_train = df_train.ffill().bfill()
df_test = df_test.ffill().bfill()

# Identify numerical features within the 'features' list
numerical_features_for_scaling = df_train[features].select_dtypes(include=np.number).columns.tolist()

# Handle potential placeholders (like -999 often found in hydrology data)
df_train[numerical_features_for_scaling] = df_train[numerical_features_for_scaling].clip(lower=0)
df_train[target_col] = df_train[target_col].clip(lower=0)

# Check if any NaNs remain after ffill/bfill and clipping before scaling
if df_train[numerical_features_for_scaling].isnull().values.any():
    print("Warning: NaNs still present in numerical features! Dropping rows with NaNs.")
    df_train = df_train.dropna(subset=numerical_features_for_scaling)

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit and transform training data, using only numerical features for X scaling
train_x_scaled = scaler_x.fit_transform(df_train[numerical_features_for_scaling])
train_y_scaled = scaler_y.fit_transform(df_train[[target_col]])

# Prepare windows (24h input -> 4h output)
def create_sequences(data_x, data_y, window_size=24, forecast_size=4):
    X, y = [], []
    for i in range(len(data_x) - window_size - forecast_size + 1):
        X.append(data_x[i : i + window_size])
        y.append(data_y[i + window_size : i + window_size + forecast_size].flatten())
    return np.array(X), np.array(y)

WINDOW_SIZE = 24
FORECAST_SIZE = 4

X_train, y_train = create_sequences(train_x_scaled, train_y_scaled, WINDOW_SIZE, FORECAST_SIZE)

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(FORECAST_SIZE) # Outputs t, t+1, t+2, t+3
])

# Gradient clipping stops NaNs from occurring
optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

print("Training baseline model...")
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

In [35]:
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(FORECAST_SIZE) # Outputs t, t+1, t+2, t+3
])

# Gradient clipping stops NaNs from occurring
optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

print("Training baseline model...")
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

Training baseline model...
Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0281 - mae: 0.1068 - val_loss: 6.6972e-05 - val_mae: 0.0062
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0089 - mae: 0.0615 - val_loss: 6.4186e-05 - val_mae: 0.0068
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0081 - mae: 0.0570 - val_loss: 5.7204e-05 - val_mae: 0.0067
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0072 - mae: 0.0531 - val_loss: 4.7595e-05 - val_mae: 0.0062
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0064 - mae: 0.0490 - val_loss: 3.8450e-05 - val_mae: 0.0058
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0058 - mae: 0.0466 - val_loss: 3.7535e-05 - val_mae: 0.0059
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0051 - mae: 0.0432 - val_loss: 2.3355e-05 - val_mae: 0.0046
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0046 - mae: 0.0406 - val_loss: 1.6703e-05 - val_mae: 0.0039
Epoch 9/10
5

In [45]:
all_results = []

# Process each gauge independently to keep sequences clean
for gauge in df_test['basin'].unique(): # Changed from 'gauge_id' to 'basin'
    # 1. Get data for this gauge and sort by date
    gauge_df = df_test[df_test['basin'] == gauge].sort_values('date').reset_index(drop=True) # Changed from 'gauge_id' to 'basin'

    # 2. Scale the features
    gauge_scaled = scaler_x.transform(gauge_df[numerical_features_for_scaling]) # Changed from 'features' to 'numerical_features_for_scaling'

    # 3. Create sliding windows (24h history)
    # X_val will have length: len(gauge_df) - WINDOW_SIZE - FORECAST_SIZE + 1
    X_val, _ = create_sequences(gauge_scaled, np.zeros((len(gauge_scaled), 1)), WINDOW_SIZE, FORECAST_SIZE)

    if len(X_val) == 0:
        continue

    # 4. Predict
    preds_scaled = model.predict(X_val)
    preds_actual = scaler_y.inverse_transform(preds_scaled)
    preds_actual = np.clip(preds_actual, 0, None) # Re-added: ensures predictions are non-negative

    # 5. Create a DataFrame for these predictions
    res = pd.DataFrame(preds_actual, columns=['Q_pred_t', 'Q_pred_t1', 'Q_pred_t2', 'Q_pred_t3'])

    # IMPORTANT: The first prediction (index 0) corresponds to the time at gauge_df index [WINDOW_SIZE]
    # We grab the corresponding dates and basin IDs for the predictions
    prediction_start_index = WINDOW_SIZE
    prediction_end_index = WINDOW_SIZE + len(preds_actual)

    prediction_dates = gauge_df['date'].iloc[prediction_start_index : prediction_end_index]
    prediction_basins = gauge_df['basin'].iloc[prediction_start_index : prediction_end_index]

    # Create the 'Unique_ID' in the required format 'YYYY-MM-DD HH:MM:SS_BASIN_ID'
    # Ensure basin IDs are zero-padded to match the example submission format
    res['Unique_ID'] = prediction_dates.dt.strftime('%Y-%m-%d %H:%M:%S') + '_' + prediction_basins.astype(str).str.zfill(8)

    all_results.append(res)

# Combine all gauge predictions into one master list
final_preds_df = pd.concat(all_results).reset_index(drop=True)

276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
547/547 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [46]:
# 1. Use the ExampleSubmission as the index
submission = example_sub[['Unique_ID']].copy()

# 2. Merge your AI's predictions onto the submission IDs
submission = submission.merge(final_preds_df, on='Unique_ID', how='left')

# 3. Fill any missing hours with 0 (safety measure)
submission = submission.fillna(0)

# 4. Save
submission.to_csv('submission_2018.csv', index=False)
print("Finished! Download 'submission_2018.csv' for the competition.")

print("--- Final Check ---")
print(f"Submission rows: {len(submission)}")
print(f"First ID: {submission['Unique_ID'].iloc[0]}") # Should be 2018-01-01...
print(f"Last ID: {submission['Unique_ID'].iloc[-1]}")  # Should be 2018-12-31...
print("Done! Your file is ready for upload.")

Finished! Download 'submission_2018.csv' for the competition.
--- Final Check ---
Submission rows: 52560
First ID: 2018-01-01 00:00:00_08178880
Last ID: 2018-12-31 23:00:00_08198500
Done! Your file is ready for upload.
